In [1]:
import numpy as np
import tensorflow as tf

In [2]:
npz_train = np.load('data_train.npz', allow_pickle=True)
train_inputs = npz_train['inputs'].astype(np.float16)
train_targets = npz_train['targets'].astype(np.int16)

npz_valid = np.load('data_validation.npz', allow_pickle=True)
valid_inputs = npz_valid['inputs'].astype(np.float16)
valid_targets = npz_valid['targets'].astype(np.int16)

npz_test = np.load('data_validation.npz', allow_pickle=True)
test_inputs = npz_test['inputs'].astype(np.float16)
test_targets = npz_test['targets'].astype(np.int16)

In [22]:
inputs_size = train_inputs.shape[1]
outputs_size = 1
nb_units = 6

model = tf.keras.models.Sequential([
    tf.keras.layers.Input(shape=(inputs_size,)),
    tf.keras.layers.Dense(nb_units, activation='relu'),
    tf.keras.layers.Dense(nb_units, activation='relu'),
    tf.keras.layers.Dense(nb_units, activation='relu'),
    tf.keras.layers.Dense(outputs_size, activation='sigmoid')
])

optimizer = tf.keras.optimizers.Adam(learning_rate=0.0005)
model.compile(optimizer=optimizer, loss='binary_crossentropy', metrics=['accuracy'])

In [23]:
max_epoch = 100
max_batch = 30
early_stop = tf.keras.callbacks.EarlyStopping(patience=2 , restore_best_weights=True)
#lr_scheduler = tf.keras.callbacks.ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=3, min_lr=1e-6)
#class_weights = {0: 1,1: 1.2}
model.fit(
    train_inputs,
    train_targets,
    batch_size = max_batch,
    epochs = max_epoch,
    #callbacks = [early_stop], #, lr_scheduler
    #class_weight=class_weights,
    validation_data = (valid_inputs, valid_targets),
    verbose = 2
)

Epoch 1/100
109/109 - 3s - 24ms/step - accuracy: 0.5183 - loss: 0.6947 - val_accuracy: 0.5184 - val_loss: 0.6915
Epoch 2/100
109/109 - 0s - 4ms/step - accuracy: 0.5425 - loss: 0.6805 - val_accuracy: 0.5651 - val_loss: 0.6784
Epoch 3/100
109/109 - 0s - 4ms/step - accuracy: 0.5658 - loss: 0.6686 - val_accuracy: 0.5848 - val_loss: 0.6670
Epoch 4/100
109/109 - 0s - 4ms/step - accuracy: 0.5937 - loss: 0.6577 - val_accuracy: 0.6265 - val_loss: 0.6555
Epoch 5/100
109/109 - 0s - 4ms/step - accuracy: 0.6217 - loss: 0.6484 - val_accuracy: 0.6486 - val_loss: 0.6464
Epoch 6/100
109/109 - 0s - 4ms/step - accuracy: 0.6358 - loss: 0.6398 - val_accuracy: 0.6609 - val_loss: 0.6370
Epoch 7/100
109/109 - 0s - 3ms/step - accuracy: 0.6493 - loss: 0.6317 - val_accuracy: 0.6781 - val_loss: 0.6277
Epoch 8/100
109/109 - 0s - 4ms/step - accuracy: 0.6591 - loss: 0.6245 - val_accuracy: 0.6830 - val_loss: 0.6202
Epoch 9/100
109/109 - 0s - 4ms/step - accuracy: 0.6708 - loss: 0.6178 - val_accuracy: 0.7027 - val_loss

In [24]:
from sklearn.metrics import confusion_matrix, classification_report

# Evaluate model
test_loss, test_accuracy = model.evaluate(test_inputs, test_targets)
print(f'Test loss : {test_loss:.2f} -- Test accuracy : {test_accuracy:.4f}')

# Predict probabilities
predictions = model.predict(test_inputs)

# Convert probabilities → class labels
pred_targets = (predictions > 0.5).astype(int).flatten()

# Metrics
print("\nConfusion Matrix:")
print(confusion_matrix(test_targets, pred_targets))

print("\nClassification Report:")
print(classification_report(test_targets, pred_targets))

13/13 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.7912 - loss: 0.4506 
Test loss : 0.45 -- Test accuracy : 0.7912
13/13 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step

Confusion Matrix:
[[171  35]
 [ 50 151]]

Classification Report:
              precision    recall  f1-score   support

           0       0.77      0.83      0.80       206
           1       0.81      0.75      0.78       201

    accuracy                           0.79       407
   macro avg       0.79      0.79      0.79       407
weighted avg       0.79      0.79      0.79       407

